# Hiver SDE Intern — AppleSupport AI Support Agent

A compact submission notebook for the Hiver take-home. The agent classifies customer messages, retrieves similar historical AppleSupport resolutions, drafts a grounded reply, and decides AUTO-HANDLE vs ESCALATE.

## 1. Setup

In [ ]:
!pip -q install sentence-transformers faiss-cpu scikit-learn pandas requests huggingface_hub

import os, json, zipfile, requests
import numpy as np
import pandas as pd
from getpass import getpass
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, cohen_kappa_score
from sklearn.base import clone
pd.set_option("display.max_colwidth", 300)

## 2. Load Twitter data and build AppleSupport pairs

Upload `archive.zip`. Only AppleSupport support replies with a linked customer tweet are used.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_path = next(iter(uploaded))
extract_path = "/content/twitter_data"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

df = pd.read_csv("/content/twitter_data/twcs/twcs.csv")
tweet_lookup = df.set_index("tweet_id")
support_replies = df[(df.author_id == "AppleSupport") & (df.inbound == False) & df.in_response_to_tweet_id.notna()].copy()
support_replies["customer_text"] = support_replies.in_response_to_tweet_id.map(tweet_lookup.text)
support_replies["customer_id"] = support_replies.in_response_to_tweet_id.map(tweet_lookup.author_id)
support_pairs = support_replies[support_replies.customer_text.notna()].copy()
print("Full dataset:", df.shape)
print("Valid AppleSupport pairs:", len(support_pairs))

## 3. Golden evaluation set

Upload `golden_set_200_labeled.csv`. It contains 200 hand-labelled AppleSupport customer messages sampled with `random_state=42`.

In [ ]:
uploaded = files.upload()
golden = pd.read_csv(next(iter(uploaded)))
VALID_INTENTS = ["Software / iOS Update","Device / Hardware Problem","Battery / Charging","App Problem","Apple Services","Connectivity","Messaging / Notifications","Account / Apple ID","Purchase / Activation","Payment / Billing","Complaint / Feedback","Other / Unclear"]
assert len(golden) == 200 and golden.intent.isin(VALID_INTENTS).all()
print(golden.intent.value_counts())

### Intent taxonomy

In [ ]:
INTENT_DEFINITIONS = {
"Software / iOS Update":"iOS/macOS updates, installation failures, update-related software problems",
"Device / Hardware Problem":"device malfunction, physical hardware, sound, screen, camera problems",
"Battery / Charging":"battery drain, battery health, charging, charger or power problems",
"App Problem":"an app not working, crashing, loading or downloading incorrectly",
"Apple Services":"iCloud, Find My, Apple Music, Podcasts, Siri and other Apple services",
"Connectivity":"Wi-Fi, cellular data, Bluetooth or network connection problems",
"Messaging / Notifications":"Messages, iMessage, SMS or notification problems",
"Account / Apple ID":"Apple ID, login, password or account access problems",
"Purchase / Activation":"device purchase, activation, setup or activation-lock problems",
"Payment / Billing":"charges, billing or payment-method problems",
"Complaint / Feedback":"complaints, criticism, feature feedback or dissatisfaction",
"Other / Unclear":"insufficient context or issue outside the taxonomy"}

## 4. Reproducible 160/40 train-test split

In [ ]:
payment = golden[golden.intent == "Payment / Billing"]
other = golden[golden.intent != "Payment / Billing"]
train_other, test = train_test_split(other, test_size=40, random_state=42, stratify=other.intent)
train = pd.concat([train_other, payment], ignore_index=True)
X_train, y_train = train.customer_text.fillna(""), train.intent
X_test, y_test = test.customer_text.fillna(""), test.intent
print("Train:", len(train), "Test:", len(test))

## 5. Compare 7 intent classifiers

The majority classifier is the trivial baseline. Six additional classical models are benchmarked against the existing Logistic Regression approach. Macro-F1 is the main selection metric because the classes are imbalanced.

In [ ]:
def tfidf():
    return TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=2, max_df=.95, sublinear_tf=True)

model_specs = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Linear SVM": LinearSVC(class_weight="balanced"),
    "Multinomial NB": MultinomialNB(alpha=1.0),
    "Complement NB": ComplementNB(alpha=1.0),
    "SGD Classifier": SGDClassifier(loss="hinge", max_iter=2000, class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
}
trained_models, rows = {}, []
for name, clf in model_specs.items():
    pipe = Pipeline([("tfidf", tfidf()), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    trained_models[name] = pipe
    rows.append({"model":name, "accuracy":accuracy_score(y_test,pred), "macro_f1":f1_score(y_test,pred,average="macro",zero_division=0)})
model_comparison = pd.DataFrame(rows).sort_values(["macro_f1","accuracy"], ascending=False).reset_index(drop=True)
model_comparison

### Production-model decision

In the observed run, Ridge Classifier produced the best Macro-F1 (**0.285**) but Logistic Regression is retained for the support agent because it provides class probabilities needed for conservative safety gating.

In [ ]:
baseline_model = trained_models["Logistic Regression"]
majority_intent = y_train.value_counts().idxmax()
majority_pred = [majority_intent] * len(y_test)
print("Majority baseline:", round(accuracy_score(y_test,majority_pred),4), round(f1_score(y_test,majority_pred,average="macro",zero_division=0),4))
print("Logistic Regression:", round(accuracy_score(y_test,baseline_model.predict(X_test)),4), round(f1_score(y_test,baseline_model.predict(X_test),average="macro"),4))
print(classification_report(y_test, baseline_model.predict(X_test), zero_division=0))

## 6. SBERT + FAISS historical retrieval

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

golden_ids = set(golden.tweet_id.astype(str))
retrieval_data = support_pairs[~support_pairs.tweet_id.astype(str).isin(golden_ids)].sample(n=min(30000,len(support_pairs)),random_state=42).reset_index(drop=True)
retrieval_data["predicted_intent"] = baseline_model.predict(retrieval_data.customer_text.fillna(""))
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
retrieval_embeddings = embedding_model.encode(retrieval_data.customer_text.fillna("").tolist(),batch_size=64,normalize_embeddings=True,show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(retrieval_embeddings.shape[1]); index.add(retrieval_embeddings)
print("Indexed historical cases:", index.ntotal)

In [ ]:
def retrieve_similar_cases(message,k=5):
    pred = baseline_model.predict([message])[0]
    candidates = np.where((retrieval_data.predicted_intent == pred).values)[0]
    if len(candidates) < k: candidates = np.arange(len(retrieval_data))
    q = embedding_model.encode([message],normalize_embeddings=True)[0].astype("float32")
    scores = np.dot(retrieval_embeddings[candidates],q)
    pos = np.argsort(scores)[::-1][:k]; idxs = candidates[pos]
    out = retrieval_data.iloc[idxs].copy(); out["similarity"] = scores[pos]
    return pred,out[["customer_text","text","predicted_intent","similarity"]]

def build_evidence(message,k=5):
    pred, results = retrieve_similar_cases(message,k)
    evidence=[{"customer_issue":r.customer_text,"historical_reply":r.text,"similarity":float(r.similarity)} for _,r in results.iterrows()]
    return pred,evidence

## 7. Grounded reply generation

In [ ]:
def create_support_prompt(message,predicted_intent,evidence):
    evidence_text="".join([f"\nHistorical Case {i}\nCustomer: {e['customer_issue']}\nAppleSupport: {e['historical_reply']}\nSimilarity: {e['similarity']:.3f}\n" for i,e in enumerate(evidence,1)])
    return f"""You are an AI customer-support assistant for AppleSupport.
Customer message: {message}
Predicted intent: {predicted_intent}
Historical cases:{evidence_text}
Draft a concise customer-facing reply. Base it on the evidence. Do not invent policies, refunds, guarantees, troubleshooting steps, technical claims or URLs. If evidence is insufficient, ask a useful clarifying question. Return only the reply."""

In [ ]:
GEMINI_API_KEY = getpass("Enter Gemini API key: ")
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent"
def generate_with_gemini(prompt):
    r=requests.post(GEMINI_URL,headers={"Content-Type":"application/json","x-goog-api-key":GEMINI_API_KEY},json={"contents":[{"parts":[{"text":prompt}]}]})
    if r.status_code!=200: return None
    try: return r.json()["candidates"][0]["content"]["parts"][0]["text"]
    except: return None

## 8. Conservative auto-handle policy

In [ ]:
# Training-only CV threshold selection
cv_data=train[train.intent!="Payment / Billing"]
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
cv_prob=cross_val_predict(clone(baseline_model),cv_data.customer_text,cv_data.intent,cv=cv,method="predict_proba")
cv_pred=baseline_model.classes_[np.argmax(cv_prob,axis=1)] if hasattr(baseline_model,"classes_") else None
# Refit a clone to get class ordering used by CV
cv_model=clone(baseline_model); cv_model.fit(cv_data.customer_text,cv_data.intent)
cv_pred=cv_model.classes_[np.argmax(cv_prob,axis=1)]
cv_conf=np.max(cv_prob,axis=1); cv_correct=cv_pred==cv_data.intent.values
for t in [0.15,0.16,0.17,0.18,0.19,0.20,0.21,0.22]:
    m=cv_conf>=t
    if m.any(): print(t,"coverage",round(m.mean(),3),"auto accuracy",round(cv_correct[m].mean(),3),"n",m.sum())
CONFIDENCE_THRESHOLD=0.21
SIMILARITY_THRESHOLD=0.60

In [ ]:
def decide_handling(predicted_intent,evidence,message):
    if predicted_intent=="Other / Unclear": return {"decision":"ESCALATE","reason":"Intent is unclear, so automated handling is unsafe."}
    confidence=float(np.max(baseline_model.predict_proba([message])[0]))
    sims=[e["similarity"] for e in evidence]; top=max(sims,default=0); strong=sum(s>=SIMILARITY_THRESHOLD for s in sims)
    if confidence<CONFIDENCE_THRESHOLD: return {"decision":"ESCALATE","reason":f"Classifier confidence is low ({confidence:.3f})."}
    if top<SIMILARITY_THRESHOLD: return {"decision":"ESCALATE","reason":f"Historical evidence is weak (top similarity={top:.3f})."}
    if strong<2: return {"decision":"ESCALATE","reason":"There are not enough strong historical matches to safely automate the response."}
    return {"decision":"AUTO-HANDLE","reason":f"Classifier confidence ({confidence:.3f}) and historical evidence ({top:.3f}) meet the safety thresholds."}

## 9. End-to-end agent

In [ ]:
def apple_support_agent(message,k=5,generate_reply=True):
    pred,evidence=build_evidence(message,k)
    decision=decide_handling(pred,evidence,message)
    reply=None
    if generate_reply: reply=generate_with_gemini(create_support_prompt(message,pred,evidence))
    return {"intent":pred,"reply":reply,"decision":decision["decision"],"reason":decision["reason"],"evidence":evidence}

print(apple_support_agent("My iPhone battery drains very quickly after the update.",generate_reply=False))

## 10. Final 40-example safety evaluation

In [ ]:
evaluation=[]
for _,row in test.iterrows():
    pred,evidence=build_evidence(row.customer_text,5); d=decide_handling(pred,evidence,row.customer_text)
    evaluation.append({"true_intent":row.intent,"predicted_intent":pred,"decision":d["decision"],"reason":d["reason"],"top_similarity":max([e["similarity"] for e in evidence],default=0)})
evaluation_df=pd.DataFrame(evaluation)
wrong_auto=evaluation_df[(evaluation_df.decision=="AUTO-HANDLE")&(evaluation_df.true_intent!=evaluation_df.predicted_intent)]
print("Intent accuracy:",round((evaluation_df.true_intent==evaluation_df.predicted_intent).mean(),4))
print("AUTO-HANDLE:",(evaluation_df.decision=="AUTO-HANDLE").sum())
print("ESCALATE:",(evaluation_df.decision=="ESCALATE").sum())
print("Wrong AUTO-HANDLE:",len(wrong_auto))

## 11. Reply-quality LLM judge

In [ ]:
from huggingface_hub import InferenceClient
HF_TOKEN=getpass("Enter Hugging Face token for the judge: ")
hf_client=InferenceClient(provider="ovhcloud",api_key=HF_TOKEN)
JUDGE_MODEL="Qwen/Qwen3.8-27B"

def judge_reply_hf(message,reply,evidence):
    prompt=f"""Evaluate this AppleSupport reply.
CUSTOMER: {message}
REPLY: {reply}
EVIDENCE: {evidence}
Score 0-2: relevance, groundedness, helpfulness, professionalism, hallucination-free. Overall is the sum. Return ONLY JSON with keys relevance, groundedness, helpfulness, professionalism, hallucination, overall_score, reason."""
    try:
        r=hf_client.chat.completions.create(model=JUDGE_MODEL,messages=[{"role":"user","content":prompt}],max_tokens=2000,temperature=0)
        text=r.choices[0].message.content
        if text is None:return None
        return json.loads(text.strip().replace("```json","").replace("```","").strip())
    except Exception as e:return {"error":str(e)}

In [ ]:
reply_rows=[]
for _,row in test.head(10).iterrows():
    pred,evidence=build_evidence(row.customer_text,5)
    reply=generate_with_gemini(create_support_prompt(row.customer_text,pred,evidence))
    d=decide_handling(pred,evidence,row.customer_text)
    reply_rows.append({"customer_message":row.customer_text,"true_intent":row.intent,"predicted_intent":pred,"reply":reply,"decision":d["decision"],"evidence":evidence})
reply_df=pd.DataFrame(reply_rows)
reply_df[["true_intent","predicted_intent","decision","reply"]]

In [ ]:
judge_rows=[]
for _,row in reply_df.iterrows():
    s=judge_reply_hf(row.customer_message,row.reply,row.evidence)
    if s and "error" not in s: judge_rows.append(s)
judge_df=pd.DataFrame(judge_rows)
for c in ["relevance","groundedness","helpfulness","professionalism","hallucination","overall_score"]: print(c,round(judge_df[c].mean(),3))

### Human agreement

Independently score the same 10 replies using the same 0–2 rubric. The customer message is shown so the human can judge usefulness correctly.

In [ ]:
human=[]
for i,row in reply_df.iterrows():
    print("\nCASE",i+1); print("CUSTOMER:",row.customer_message); print("REPLY:",row.reply)
    vals={"relevance":int(input("Relevance (0-2): ")),"groundedness":int(input("Groundedness (0-2): ")),"helpfulness":int(input("Helpfulness (0-2): ")),"professionalism":int(input("Professionalism (0-2): ")),"hallucination":int(input("Hallucination-free (0-2): "))}
    vals["overall_score"]=sum(vals.values()); human.append(vals)
human_df=pd.DataFrame(human)

In [ ]:
for c in ["relevance","groundedness","helpfulness","professionalism","hallucination"]:
    exact=np.mean(judge_df[c].astype(int).values==human_df[c].astype(int).values)
    k=cohen_kappa_score(judge_df[c].astype(int),human_df[c].astype(int),weights="quadratic")
    print(c,"exact agreement:",f"{exact:.1%}","quadratic kappa:",round(k,3))
mae=np.mean(np.abs(judge_df.overall_score.astype(int)-human_df.overall_score.astype(int)))
print("Overall score MAE:",round(mae,3))

## 12. Submission notes

### Observed results
- Majority baseline: **30.0% accuracy / 0.042 Macro-F1**.
- TF-IDF + Logistic Regression: **37.5% / 0.243 Macro-F1**.
- Best additional classical model observed: **Ridge Classifier, 40.0% / 0.285 Macro-F1**.
- Conservative safety test: **1/40 AUTO-HANDLE, 39/40 ESCALATE, 0 wrong AUTO-HANDLE**.
- 10-case reply audit: **7.5/10 overall**, with helpfulness the weakest dimension at **0.9/2**.
- 10-case judge-human audit: relevance 60%, groundedness 50%, helpfulness 20%, professionalism 50%, hallucination-free 90%; overall-score MAE **1.3/10**.

### Key failure modes
1. Intent ambiguity, especially update vs battery/device/app issues.
2. High semantic similarity can still retrieve the wrong issue.
3. Generated replies can be too generic.
4. Historical support links can be repeated even when not useful.
5. Rare intents have too little labelled data.

### Misleading headline number
The 0% wrong-auto-handle rate is based on only 40 held-out examples and an intentionally conservative policy that auto-handled only 1 case. It demonstrates a safety boundary, not broad automation capability.

### One more week
Expand and double-label the golden set, calibrate confidence, rerank retrieved resolutions, add a reply-quality gate, and test on a larger time-separated set.

### Decision log
The main non-obvious decisions were: AppleSupport selection; 12-intent taxonomy; 200-example golden set; 40-example holdout; keeping the one Payment/Billing example in training; Macro-F1 for imbalance; excluding golden examples from retrieval; 30k retrieval subsample; intent-aware retrieval; Logistic Regression for probability-based safety gating; conservative auto-handling; training-only CV threshold selection; separate Qwen judge; and explicit judge-human audit.